In [3]:
# Import Dataset
from google.colab import files
uploaded = files.upload()
import sqlite3
import pandas as pd
df = pd.read_csv('logistics_dataset.csv')

# Dataset Memory
conn = sqlite3.connect(':memory:')
df.to_sql('shipment', conn, index=False, if_exists='replace')
print(f"Data berhasil diimport: {len(df)} baris")

Saving logistics_dataset.csv to logistics_dataset (1).csv
Data berhasil diimport: 1500 baris


In [5]:
# query1 : Lihat 5 data teratas
query1 = "SELECT * FROM shipment LIMIT 5"
pd.read_sql(query1, conn)

,shipment_id,date,supplier,warehouse,destination,route,transport_mode,driver,distance_km,fuel_cost,...,total_cost,revenue,profit,delivery_time_hours,delivery_status,month,cost_per_km,profit_margin_pct,vehicle,is_delayed
0,SHP-001169,8/6/2024,Orion Freight Co.,East Depot - Sylhet,Jessore Commerce Center,Sylhet → Jessore Commerce Center,Light Van,A. Karim,71.2,57.39,...,150,200,114,1.1,On Time,2025-08,2.066011,36.331371,Light Van - A.Karim,0
1,SHP-001346,8/7/2024,Summit Cargo Partners,East Depot - Sylhet,Rajshahi Industrial Zone,Sylhet → Rajshahi Industrial Zone,Refrigerated Truck,R. Chowdhury,38.6,73.10,...,150,200,114,1.9,On Time,2025-08,5.436788,29.173135,Refrigerated Truck - R.Chowdhury,0
2,SHP-000401,8/8/2024,Meridian Shipping Ltd.,West Storage - Khulna,Khulna Trade Zone,Khulna → Khulna Trade Zone,Light Van,J. Miah,75.4,50.69,...,150,200,114,3.4,On Time,2025-08,2.381167,18.705003,Light Van - J.Miah,0
3,SHP-001336,8/9/2024,Meridian Shipping Ltd.,East Depot - Sylhet,Bogura Supply Point,Sylhet → Bogura Supply Point,Light Van,S. Ahmed,148.2,119.55,...,150,200,114,3.8,On Time,2025-08,1.470850,33.907401,Light Van - S.Ahmed,0
4,SHP-000820,8/10/2024,NorthStar Carriers,West Storage - Khulna,Sylhet Market District,Khulna → Sylhet Market District,Medium Truck,S. Ahmed,167.2,209.84,...,150,200,114,3.2,On Time,2025-08,2.326615,32.064895,Medium Truck - S.Ahmed,0


In [6]:
# query2 : Filter pengiriman yang statusnya 'Delayed'
query2 = '''
SELECT shipment_id, supplier, destination, delivery_status, delivery_time_hours
FROM shipment
WHERE delivery_status = 'Delayed'
'''
pd.read_sql(query2, conn)

,shipment_id,supplier,destination,delivery_status,delivery_time_hours
0,SHP-000347,Summit Cargo Partners,Bogura Supply Point,Delayed,7.0
1,SHP-001082,Summit Cargo Partners,Bogura Supply Point,Delayed,5.3
2,SHP-000772,Falcon Transit Group,Barisal Distribution Point,Delayed,1.0
3,SHP-001198,Summit Cargo Partners,Jessore Commerce Center,Delayed,1.7
4,SHP-000914,Meridian Shipping Ltd.,Bogura Supply Point,Delayed,2.0
...,...,...,...,...,...
294,SHP-000147,Falcon Transit Group,Mymensingh Depot,Delayed,2.5
295,SHP-000429,BlueWave Logistics,Sylhet Market District,Delayed,9.4
296,SHP-000170,Atlas Haulage,Barisal Distribution Point,Delayed,1.7
297,SHP-000540,Summit Cargo Partners,Sylhet Market District,Delayed,8.4


In [7]:
# query3: Cari 10 pengiriman dengan profit tertinggi
query3 = '''
SELECT shipment_id, supplier, destination, revenue, profit, profit_margin_pct
FROM shipment
ORDER BY profit DESC
LIMIT 10
'''
pd.read_sql(query3, conn)

,shipment_id,supplier,destination,revenue,profit,profit_margin_pct
0,SHP-001169,Orion Freight Co.,Jessore Commerce Center,200,114,36.331371
1,SHP-001346,Summit Cargo Partners,Rajshahi Industrial Zone,200,114,29.173135
2,SHP-000401,Meridian Shipping Ltd.,Khulna Trade Zone,200,114,18.705003
3,SHP-001336,Meridian Shipping Ltd.,Bogura Supply Point,200,114,33.907401
4,SHP-000820,NorthStar Carriers,Sylhet Market District,200,114,32.064895
5,SHP-000511,Summit Cargo Partners,Bogura Supply Point,200,114,24.307372
6,SHP-000691,Falcon Transit Group,Mymensingh Depot,200,114,34.105999
7,SHP-000810,Meridian Shipping Ltd.,Barisal Distribution Point,200,114,33.204815
8,SHP-001117,Cobalt Route Systems,Jessore Commerce Center,200,114,22.688489
9,SHP-001422,NorthStar Carriers,Jessore Commerce Center,200,114,26.406012


In [8]:
# query4: Total dan rata-rata biaya per moda transportasi
query4 = '''
SELECT
    transport_mode,
    COUNT(*) AS jumlah_pengiriman,
    ROUND(AVG(total_cost), 2) AS rata_rata_biaya,
    ROUND(SUM(total_cost), 2) AS total_biaya
FROM shipment
GROUP BY transport_mode
ORDER BY total_biaya DESC
'''
pd.read_sql(query4, conn)

,transport_mode,jumlah_pengiriman,rata_rata_biaya,total_biaya
0,Heavy Truck,327,150.0,49050.0
1,Light Van,304,150.0,45600.0
2,Refrigerated Truck,297,150.0,44550.0
3,Medium Truck,291,150.0,43650.0
4,Trailer,281,150.0,42150.0


In [9]:
# query5: Persentase keterlambatan per supplier
query5 = '''
SELECT
    supplier,
    COUNT(*) AS total_pengiriman,
    SUM(CASE WHEN is_delayed = 1 THEN 1 ELSE 0 END) AS jumlah_delay,
    ROUND(100.0 * SUM(CASE WHEN is_delayed = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS persen_delay
FROM shipment
GROUP BY supplier
ORDER BY persen_delay DESC
'''
pd.read_sql(query5, conn)

,supplier,total_pengiriman,jumlah_delay,persen_delay
0,Summit Cargo Partners,194,49,25.26
1,Meridian Shipping Ltd.,184,43,23.37
2,BlueWave Logistics,210,46,21.90
3,NorthStar Carriers,182,35,19.23
4,Orion Freight Co.,178,32,17.98
5,Cobalt Route Systems,197,34,17.26
6,Atlas Haulage,169,29,17.16
7,Falcon Transit Group,186,31,16.67
